In [6]:
# Setting up Global Config 

unzip_wallet = False                 # For zip wallet file
create_tables = False                # for build schema (skip creating table if already table exists)
empty_tables = False                 # Truncate the raws (safe reset for table for rerunning the files)
drop_tables = False                  # for only a destructive clean slate of DB

In [7]:
import oracledb
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
wallet_zip_name = os.getenv("wallet_zip_name")
wallet_password = os.getenv("wallet_password")
wallet_path_from_env = os.getenv("wallet_path")
wallet_path = os.path.abspath(wallet_path_from_env)
user_from_env = os.getenv("user")
dsn_from_env = os.getenv("dsn")

In [9]:
# Setting up the connection
try:
    print("Attempting secure MTS Connection")

    connection = oracledb.connect(
        user = user_from_env,
        password = wallet_password,
        dsn = dsn_from_env,
        config_dir = wallet_path,
        wallet_location = wallet_path,
        wallet_password = wallet_password
    )

    cursor = connection.cursor()
    cursor.execute("SELECT 'Connected!' FROM dual")
    print(cursor.fetchone())

except oracledb.Error as e:
    print("\nStill hitting a wall. Here is the exact error:")
    print(e)

Attempting secure MTS Connection
('Connected!',)


In [ ]:
"""
    Initializing the HR Recruitment Schema
    Candidate Pool :- The Hybrid table (Structured + Vector)
"""


# Please create_table flag here 


cursor.execute("""
    CREATE TABLE candidate_pool (
        candidate_id VARCHAR(50) PRIMARY KEY,
        full_name VARCHAR2(100),
        summary CLOB ,                                  -- The text we will Vectorize
        skills VARCHAR2(1000),                          -- Comma-Seprated list of Keywords
        years_experience NUMBER,                        -- For SQL Filtering (e.g. > 5 years)
        salary_expectation NUMBER,                      -- For SQL Filtering (e.g. < 120k)
        resume_vector VECTOR(1536)                      -- Semantic Brain
    )
""")
print("Table Candidate_pool Created.")

Table Candidate_pool Created.


In [12]:
""" 
    Recruitment_Rules : Domain-Specific Instructions
    Seperate this from the Generic Context_library to show domain isolation
"""

cursor.execute("""
    CREATE TABLE recruitment_rules (
        rule_id VARCHAR2(50) PRIMARY KEY,
        agent_persona CLOB,                         -- eg. "Culture fit officer" vs "Technical Screener
        evaluation_criteria CLOB,                   -- Specific rubric for agent
        rule_vector VECTOR(1536)
    )
""")

print("Table RECRUITMENT_RULES Created")

Table RECRUITMENT_RULES Created


In [13]:
connection.commit()
print("HR Schema Initialized successfully")

HR Schema Initialized successfully


In [18]:
# Checking the Tables 

## Checking Existance of Tables 
cursor.execute(""" 
    SELECT table_name
    from user_tables 
    where table_name in ('CANDIDATE_POOL' , 'RECRUITMENT_RULES')
    order by table_name   
""")

tables = cursor.fetchall()
print(f"Tables found {len(tables)}")

Tables found 2


In [22]:
print(tables)

[('CANDIDATE_POOL',), ('RECRUITMENT_RULES',)]


In [26]:
# Inspect the Candidate_pool Table 
print("Column Defination of the candidate_pool Table")
cursor.execute("""
    select column_name , data_type , data_length
    from user_tab_columns
    where table_name = 'CANDIDATE_POOL'
    order by column_id
""")

for row in cursor.fetchall():
    print(row)


# Inspect the Recruitment_rules Table
print("")
print("*"*50) 
print("\nColumn Defination of the Recruitment_rules Table")
cursor.execute("""
    select column_name , data_type , data_length
    from user_tab_columns
    where table_name = 'RECRUITMENT_RULES'
    order by column_id
""")

for row in cursor.fetchall():
    print(row)


# connection Heartbeat
print("")
print("*"*50)
print("\nRunning a simple test query on DUAL...")
cursor.execute("select 'Oracle connection OK' from dual")
print(cursor.fetchone()[0])

Column Defination of the candidate_pool Table
('CANDIDATE_ID', 'VARCHAR2', 50)
('FULL_NAME', 'VARCHAR2', 100)
('SUMMARY', 'CLOB', 4000)
('SKILLS', 'VARCHAR2', 1000)
('YEARS_EXPERIENCE', 'NUMBER', 22)
('SALARY_EXPECTATION', 'NUMBER', 22)
('RESUME_VECTOR', 'VECTOR', 8200)

**************************************************

Column Defination of the Recruitment_rules Table
('RULE_ID', 'VARCHAR2', 50)
('AGENT_PERSONA', 'CLOB', 4000)
('EVALUATION_CRITERIA', 'CLOB', 4000)
('RULE_VECTOR', 'VECTOR', 8200)

**************************************************

Running a simple test query on DUAL...
Oracle connection OK


In [28]:
def empty_vector_tables(cursor):
    """
        Empty all the tables using truncate not deleting the tables
    """
    try:
        print("Empting the candidate_pool")
        cursor.execute("truncate table CANDIDATE_POOL")

        print("Empting recruitment_rules")
        cursor.execute("truncate table RECRUITMENT_RULES")
    except Exception as e:
        print(f"Error emptying the tables : {str(e)}")


empty_tables = True
if empty_tables:
    empty_vector_tables(cursor=cursor)
empty_tables = False    

Empting the candidate_pool
Empting recruitment_rules
